In [1]:
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
import torch
import evaluate


In [2]:
LLM_MODEL = "distilbert-base-uncased"

In [3]:

# Data loading + preprocessing
csv_path = "data/train.csv"
df = pd.read_csv(csv_path)

# 1) Label encoding
if set(["winner_model_a", "winner_model_b", "winner_tie"]).issubset(df.columns):
    df["labels"] = (
        df["winner_model_a"] * 0 +
        df["winner_model_b"] * 1 +
        df["winner_tie"] * 2
    ).astype("int64")
else:
    raise ValueError("Missing one of winner_model_a/winner_model_b/winner_tie")

# 2) Build text field
if set(["prompt", "response_a", "response_b"]).issubset(df.columns):
    df["text"] = (
        "Prompt:\n" + df["prompt"].astype(str) +
        "\n\nResponse A:\n" + df["response_a"].astype(str) +
        "\n\nResponse B:\n" + df["response_b"].astype(str)
    )
else:
    raise ValueError("Missing one of prompt/response_a/response_b")

print("Loaded", len(df), "rows")
print(df["labels"].value_counts())

Loaded 57477 rows
labels
0    20064
1    19652
2    17761
Name: count, dtype: int64


In [4]:


# 3) HF dataset
dataset = Dataset.from_pandas(df[["text", "labels"]])

# 4) split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]


def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )


tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/51729 [00:00<?, ? examples/s]

Map:   0%|          | 0/5748 [00:00<?, ? examples/s]

In [5]:


model = AutoModelForSequenceClassification.from_pretrained(
    LLM_MODEL,
    num_labels=3
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)


training_args = TrainingArguments(
    output_dir="./model",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps=1200,
    warmup_steps=0.1,
    weight_decay=0.01,
    fp16=False,
    save_strategy="steps",
    eval_strategy="steps",
    eval_steps=120,
    save_steps=480,
    logging_strategy="steps",
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    logging_nan_inf_filter=False,
    # use_cpu=True,
)


In [7]:


data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics, 
)

In [8]:
#B. Check a manual forward pass on a train batch

# Do this before full training and then again after a few steps:

name, p = next((n, p) for n, p in model.named_parameters() if p.requires_grad)
before = p.detach().cpu().clone()

trainer.train()

after = dict(model.named_parameters())[name].detach().cpu()
print("param changed?", not torch.equal(before, after))
print("max abs diff:", (after - before).abs().max().item())

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Accuracy
120,4.361400,1.099869,0.345685
240,4.368452,1.090352,0.367432
360,4.430029,1.091946,0.361169
480,4.318280,1.093332,0.366736
600,4.319626,1.086914,0.372129
720,4.306258,1.089867,0.378740
840,4.331231,1.086207,0.398573
960,4.403927,1.084151,0.398747
1080,4.363099,1.082666,0.401531
1200,4.381819,1.081436,0.403619


/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-cla

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


param changed? True
max abs diff: 0.002176114358007908


In [11]:
trainer.remove_callback(type(trainer.callback_handler.callbacks[-1]))  # not ideal/generic

In [12]:
print("global_step:", trainer.state.global_step)
print(trainer.state.log_history[-10:])
print("val metrics:", trainer.evaluate(val_dataset))

pred = trainer.predict(val_dataset)
print("nan/inf", np.any(np.isnan(pred.predictions)), np.any(np.isinf(pred.predictions)))

global_step: 1200
[{'eval_loss': 1.0826661586761475, 'eval_accuracy': 0.40153096729297144, 'eval_runtime': 27.1207, 'eval_samples_per_second': 211.942, 'eval_steps_per_second': 26.511, 'epoch': 0.16702107094529287, 'step': 1080}, {'loss': 4.347348022460937, 'grad_norm': 2.3410325050354004, 'learning_rate': 1.8703703703703705e-06, 'epoch': 0.17011405374057606, 'step': 1100}, {'loss': 4.308596801757813, 'grad_norm': 2.3758833408355713, 'learning_rate': 1.4074074074074075e-06, 'epoch': 0.17398028223468007, 'step': 1125}, {'loss': 4.322787780761718, 'grad_norm': 2.280388593673706, 'learning_rate': 9.444444444444445e-07, 'epoch': 0.17784651072878407, 'step': 1150}, {'loss': 4.397773742675781, 'grad_norm': 2.3554203510284424, 'learning_rate': 4.814814814814815e-07, 'epoch': 0.18171273922288808, 'step': 1175}, {'loss': 4.38181884765625, 'grad_norm': 1.8157340288162231, 'learning_rate': 1.851851851851852e-08, 'epoch': 0.18557896771699206, 'step': 1200}, {'eval_loss': 1.0814357995986938, 'eval_

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


val metrics: {'eval_loss': 1.0814357995986938, 'eval_accuracy': 0.4036186499652053, 'eval_runtime': 27.3871, 'eval_samples_per_second': 209.88, 'eval_steps_per_second': 26.253, 'epoch': 0.18557896771699206}


/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


nan/inf False False


In [13]:
# Save the trained model and tokenizer
trainer.save_model("./model")
tokenizer.save_pretrained("./model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model/tokenizer_config.json', './model/tokenizer.json')

In [14]:

# Evaluate and show metrics
metrics = trainer.evaluate(eval_dataset=val_dataset)
print("Validation metrics:", metrics)

# Predict to verify output shape
predictions = trainer.predict(val_dataset)
pred_labels = predictions.predictions.argmax(axis=-1)
print("Sample predictions:", pred_labels[:10])

/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Validation metrics: {'eval_loss': 1.0814357995986938, 'eval_accuracy': 0.4036186499652053, 'eval_runtime': 27.167, 'eval_samples_per_second': 211.58, 'eval_steps_per_second': 26.466, 'epoch': 0.18557896771699206}


/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Sample predictions: [0 0 2 1 0 2 2 1 1 1]


In [15]:
# 1) Load test data
test_df = pd.read_csv("data/test.csv")

# 2) Prepare text in same format as training
test_df["text"] = (
    "Prompt:\n" + test_df["prompt"].astype(str) +
    "\n\nResponse A:\n" + test_df["response_a"].astype(str) +
    "\n\nResponse B:\n" + test_df["response_b"].astype(str)
)

test_dataset = Dataset.from_pandas(test_df[["text"]])

test_dataset = test_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.remove_columns([c for c in test_dataset.column_names if c not in ["input_ids", "attention_mask", "text"]])



Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [ ]:
# # a) single record test check
# demo = test_df.iloc[[0]]
# tok = tokenizer(
#     demo["text"].tolist(),
#     truncation=True,
#     padding="max_length",
#     max_length=256,
#     return_tensors="pt",
# )
# print("input_ids shape", tok["input_ids"].shape)
# print("attention_mask sum", tok["attention_mask"].sum().item())

# # # b) model forward
# model.eval()
# with torch.no_grad():
#     out = model(input_ids=tok["input_ids"], attention_mask=tok["attention_mask"])
# print("logits:", out.logits)
# print("nan?", torch.isnan(out.logits).any().item(), "inf?", torch.isinf(out.logits).any().item())

In [ ]:
# model_cpu = model.to("cpu")
# tok_cpu = {k: v.to("cpu") for k,v in tok.items()}
# with torch.no_grad():
#     out = model_cpu(**tok_cpu)
# print(out.logits[:2])

In [ ]:
# baseline_model = AutoModelForSequenceClassification.from_pretrained(
#     "microsoft/deberta-v3-small", num_labels=3
# ).eval()

# demo_tok = tokenizer(
#     ["Prompt:\nhello\n\nResponse A:\nA\n\nResponse B:\nB"],
#     truncation=True, padding="max_length", max_length=256, return_tensors="pt"
# )

# with torch.no_grad():
#     baseline_out = baseline_model(**demo_tok)
# print("baseline logits:", baseline_out.logits)
# print("baseline nan?", torch.isnan(baseline_out.logits).any().item())

In [ ]:
# trained = AutoModelForSequenceClassification.from_pretrained("./model").eval()
# with torch.no_grad():
#     test_out = trained(**demo_tok)
# print("trained logits:", test_out.logits)
# print("trained nan?", torch.isnan(test_out.logits).any().item())

In [17]:


# 4) Predict
preds = trainer.predict(test_dataset)
print("Raw predictions shape:", preds)
logits = preds.predictions  # shape (N, 3)
print("Logits shape:", logits)
logits_tensor = torch.from_numpy(logits).float()

if torch.isnan(logits_tensor).any() or torch.isinf(logits_tensor).any():
    raise ValueError("Logits contain NaN or Inf; check model/prediction pipeline")

# stable softmax to avoid numeric instabilities
logits_stable = logits_tensor - logits_tensor.max(dim=-1, keepdim=True).values
exp_scores = torch.exp(logits_stable)
probs_tensor = exp_scores / exp_scores.sum(dim=-1, keepdim=True)
probs = probs_tensor.cpu().numpy()

# 5) save to submission file
out = pd.DataFrame({
    "id": test_df["id"].values,
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_model_tie": probs[:, 2],
})

out.to_csv("submission.csv", index=False)
print(out.head())


Raw predictions shape: PredictionOutput(predictions=array([[-0.19377387, -0.33394846,  0.37519777],
       [-0.15734042,  0.22029127, -0.24161457],
       [ 0.1174529 ,  0.0133178 , -0.29177618]], dtype=float32), label_ids=None, metrics={'test_runtime': 0.0433, 'test_samples_per_second': 69.341, 'test_steps_per_second': 23.114})
Logits shape: [[-0.19377387 -0.33394846  0.37519777]
 [-0.15734042  0.22029127 -0.24161457]
 [ 0.1174529   0.0133178  -0.29177618]]
        id  winner_model_a  winner_model_b  winner_model_tie
0   136060        0.275054        0.239078          0.485868
1   211333        0.296033        0.431860          0.272107
2  1233961        0.389823        0.351271          0.258906


/Users/yanxinye/github/ml-ops-notes/designing-ml-systems/llm-classification-finetuning/data/transformer/lib/python3.14/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
